# Load pruned + mm model

In [ ]:
import os 
os.environ["HF_HOME"] = "${HF_HOME}/"
import sys
sys.path.append('${REPO_ROOT}/VLM')
sys.path.append('${REPO_ROOT}/VLM/InternVL/internvl_chat')
import torch
from internvl.model.internvl_chat.builder import load_pruned_model_devel
model_path = "OpenGVLab/Mini-InternVL-Chat-4B-V1-5"
pruned_model_path = "${REPO_ROOT}/ShortGPT/prune_log/Mini-InternVL-Chat-4B-V1-5_pruned_20_50_samples/pruned_model.bin"
mm = "${REPO_ROOT}/VLM/InternVL/internvl_chat/ckpt/ft-5/checkpoint-2600/"
lora = "${REPO_ROOT}/VLM/InternVL/internvl_chat/ckpt/ft-5/checkpoint-2600/"
tokenizer,model = load_pruned_model_devel(model_path, pruned_model=pruned_model_path,mm=mm,lora=lora, torch_dtype=torch.float16)
# print the model parameters
print('Model Parameters:',sum(p.numel() for p in model.parameters()))
print('LLM Parameters:',sum(p.numel() for p in model.language_model.parameters()))
print('MLP1 Parameters:',sum(p.numel() for p in model.mlp1.parameters()))
print('Vision Model Parameters:',sum(p.numel() for p in model.vision_model.parameters()))

In [ ]:
import torch
weights = torch.load("${REPO_ROOT}/VLM/InternVL/internvl_chat/ckpt/ft-5/checkpoint-1800/pytorch_model-00002-of-00002.bin")
for key, tensor in weights.items():
    print(f"Key: {key}, Shape: {tensor.shape}")

In [ ]:
import torch
weights = torch.load("${REPO_ROOT}/VLM/InternVL/internvl_chat/ckpt/mm-5-testgpu/pytorch_model-00002-of-00002.bin")
for key, tensor in weights.items():
    if "mlp1" in key:
        print(f"Key: {key}, Shape: {tensor.shape}")

In [ ]:
def load_pruned_model(model_path, pruned_model_path=None, mm=None, lora=None, **kwargs):
    tokenizer = AutoTokenizer.from_pretrained(
            model_path, add_eos_token=False, trust_remote_code=True, use_fast=True)
    model = InternVLChatModel.from_pretrained(model_path, **kwargs)
    if pruned_model_path:
        print('Loading pruned model')
        pruned_model = torch.load(pruned_model_path, map_location='cpu')
        model.language_model.model.layers = deepcopy(pruned_model['model'].language_model.model.layers)
        for layer in model.language_model.model.layers:
            layer.self_attn.num_heads = layer.self_attn.qkv_proj.weight.data.shape[0] // (3 * layer.self_attn.head_dim)
        # For shortGPT, change the number of layers
        for i, layer in enumerate(model.language_model.model.layers):
            layer.self_attn.layer_idx = i
    print('Loaded pruned model')
    if mm:
        print('Loading mlp weights')
        weight_path_1 = os.path.join(mm, 'pytorch_model-00001-of-00002.bin')
        weight_path_2 = os.path.join(mm, 'pytorch_model-00002-of-00002.bin')
        weight_1 = torch.load(weight_path_1, map_location='cpu')
        weight_2 = torch.load(weight_path_2, map_location='cpu')
        mm_weights = {}
        for key, value in weight_1.items():
            if 'mlp1' in key:
                mm_weights[key] = value
        for key, value in weight_2.items():
            if 'mlp1' in key:
                mm_weights[key] = value
        model.mlp1.load_state_dict(mm_weights, strict=True)
        print('Loaded mlp weights')
        if lora:
            print("Warp language model with LORA")
            model.wrap_llm_lora(r=16,lora_alpha=32)
            language_model_weights = {}
            for key, value in weight_1.items():
                if 'lora' in key:
                    language_model_weights[key] = value
            for key, value in weight_2.items():
                if 'lora' in key:
                    language_model_weights[key] = value
            print('Loading lora weights')
            model.language_model.load_state_dict(language_model_weights, strict=True)
            print('Loaded lora weights')      
    print('Language Model architecture:', model.language_model.model)
    return tokenizer, model

paths = os.listdir('${REPO_ROOT}/VLM/InternVL/internvl_chat/ckpt/dist-10/checkpoint-2200')

In [ ]:
import os
paths = os.listdir('${REPO_ROOT}/VLM/InternVL/internvl_chat/ckpt/dist-10/checkpoint-2200')
paths

In [ ]:
paths